# E2-03 · Comparación de dos o más grupos: pruebas t y ANOVA

**Estadística II · Ingeniería Industrial · Universidad Industrial de Santander**  
*Versión docente (con soluciones)*

**Propósito de aprendizaje**

- Comparar las medias de dos grupos independientes eligiendo entre la t de Student, la t de Welch y Mann-Whitney según los supuestos.
- Plantear y resolver un ANOVA de un factor, verificar sus supuestos e identificar qué grupos difieren con Tukey.
- Tomar una decisión de compras, de calidad o de mantenimiento a partir de los resultados, distinguiendo significancia estadística de relevancia práctica.

**Duración sugerida:** 2 sesiones de 2 horas  
**Datos:** casos IC07 (precios de compra), IC05 (proveedores de acero) e IC03 (tiempos de reparación)

## Estructura

| Parte | Caso | Modalidad | Pregunta de negocio |
|---|---|---|---|
| 1 | IC07 · Precios de compra | guiada | ¿Comprar por contrato marco sale más barato que comprar en el mercado? |
| 2 | IC05 · Resistencia por proveedor | semi-guiada | ¿Los tres proveedores entregan acero con la misma resistencia media? |
| 3 | IC03 · Tiempos de reparación | autónoma | ¿El tiempo de reparación depende del tipo de equipo? |

En las celdas marcadas con `# Escriba su código aquí` usted completa el análisis. Las preguntas de interpretación se responden en las celdas de texto **✍️ Su respuesta**. Use α = 0,05.

> **📌 Nota para el docente**
>
> **Resultados esperados (verificados con las bases del repositorio):**
> - **IC07:** contrato marco (n = 248, media ≈ 47,7 mil $) vs. compra puntual (n = 99, media ≈ 53,3 mil $). La compra puntual es muy asimétrica (Shapiro rechaza) y su varianza es unas 8 veces mayor (F y Levene rechazan) → corresponde **Welch**; IC 95 % para la diferencia ≈ [−7,3 ; −4,0] mil $ por unidad. Mann-Whitney confirma.
> - **IC05:** las tres muestras son compatibles con normalidad, pero Pacífico tiene media menor (≈ 464 MPa) y más dispersión; Levene rechaza → reportar también ANOVA de Welch. Tukey: todas las parejas difieren, incluida Andes–Central (≈ 10 MPa), que es significativa pero pequeña en términos prácticos.
> - **IC03:** los tiempos son asimétricos a la derecha en los cuatro grupos (forma lognormal); con el logaritmo se cumple normalidad y casi se igualan las varianzas. Kruskal-Wallis y el ANOVA sobre log(tiempo) rechazan; el orden es Inyección > Prensa > Transportador > Empaque.

# 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
sns.set_theme(style="whitegrid", palette=["#00ADC6", "#FF6F49", "#25A687", "#7B6FD0"])
ALFA = 0.05

import os
from pathlib import Path

def cargar_csv(nombre, carpeta="casos_industriales/csv"):
    """Busca el archivo en la carpeta actual o en la carpeta datos/ del repositorio.
    Si no lo encuentra y se está en Google Colab, abre el diálogo para subirlo."""
    candidatos = [Path(nombre)] + [Path(p) / "datos" / carpeta / nombre for p in [".", "..", "../..", "../../.."]]
    for ruta in candidatos:
        if ruta.exists():
            print(f"Leyendo {ruta}")
            return pd.read_csv(ruta)
    try:
        from google.colab import files  # solo existe en Colab
        print(f"No encontré '{nombre}'. Súbalo desde su computador:")
        subido = files.upload()
        return pd.read_csv(next(iter(subido)))
    except ImportError:
        raise FileNotFoundError(f"No encontré '{nombre}'. Cópielo en la misma carpeta de este notebook.")

---
# Parte 1 (guiada) · ¿El contrato marco reduce el precio de compra?

El área de abastecimiento compra el mismo insumo de dos formas: con **contrato marco** (proveedor negociado) o con **compra puntual** en el mercado. Gerencia quiere saber si el contrato marco realmente reduce el precio unitario antes de renovarlo por otro año.

Sean μ_C y μ_P los precios medios con contrato y con compra puntual:

$$H_0:\ \mu_C = \mu_P \qquad H_1:\ \mu_C \neq \mu_P$$

In [ ]:
compras = cargar_csv("IC07_precios_compras.csv")
display(compras.groupby("tipo_compra")["precio_unitario_miles"].describe().round(2))
sns.boxplot(data=compras, x="tipo_compra", y="precio_unitario_miles")
sns.stripplot(data=compras, x="tipo_compra", y="precio_unitario_miles", color="#333", alpha=.25, size=3)
plt.ylabel("precio unitario (miles de $)"); plt.title("Precio unitario según modalidad de compra"); plt.show()

### 1.1 Supuestos: forma de cada grupo y varianzas

La t de Student supone normalidad (o n grande) **y varianzas iguales**. La t de Welch solo necesita lo primero. Revisamos ambos supuestos.

In [ ]:
contrato = compras.loc[compras["tipo_compra"] == "Contrato marco", "precio_unitario_miles"]
puntual  = compras.loc[compras["tipo_compra"] == "Compra puntual", "precio_unitario_miles"]

for nombre, x in [("Contrato marco", contrato), ("Compra puntual", puntual)]:
    print(f"{nombre:15s} n = {len(x):3d}  asimetría = {x.skew():5.2f}  Shapiro p = {stats.shapiro(x).pvalue:.4f}")

F = puntual.var() / contrato.var()
p_F = 2 * stats.f.sf(F, len(puntual) - 1, len(contrato) - 1)
print(f"\nF = s²_puntual / s²_contrato = {F:.2f}   p = {p_F:.4g}")
print(f"Levene (mediana): p = {stats.levene(contrato, puntual, center='median').pvalue:.4g}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(contrato, line="s", ax=ax[0]); ax[0].set_title("Q-Q contrato marco")
sm.qqplot(puntual, line="s", ax=ax[1]); ax[1].set_title("Q-Q compra puntual")
plt.tight_layout(); plt.show()

### 1.2 Prueba de medias e intervalo de confianza

Como las varianzas son claramente distintas, se usa **Welch**. A modo de comparación se muestran también Student y Mann-Whitney.

In [ ]:
welch = stats.ttest_ind(contrato, puntual, equal_var=False)
student = stats.ttest_ind(contrato, puntual, equal_var=True)
mw = stats.mannwhitneyu(contrato, puntual)
print(f"Welch:        t = {welch.statistic:.3f}   p = {welch.pvalue:.3g}")
print(f"Student:      t = {student.statistic:.3f}   p = {student.pvalue:.3g}")
print(f"Mann-Whitney: U = {mw.statistic:.0f}   p = {mw.pvalue:.3g}")

ic = welch.confidence_interval(0.95)
dif = contrato.mean() - puntual.mean()
print(f"\nDiferencia de medias (contrato − puntual) = {dif:.2f} mil $   IC 95 %: [{ic.low:.2f} ; {ic.high:.2f}]")
print(f"Ahorro relativo: {-dif / puntual.mean():.1%} del precio de compra puntual")

**Pregunta 1.1.** ¿Por qué en este caso corresponde la t de Welch y no la de Student? ¿Cambia la conclusión si se usa Student?

> **✅ Respuesta modelo (docente)**
>
> Porque la prueba F y la de Levene rechazan la igualdad de varianzas: la compra puntual tiene una dispersión unas ocho veces mayor que el contrato marco. La t de Student combinaría las dos varianzas en una sola y el error estándar quedaría mal estimado (con grupos de tamaños distintos, los p-valores pueden ser engañosos). En este caso las dos pruebas rechazan H₀, de modo que la conclusión no cambia, pero el procedimiento correcto es Welch. Mann-Whitney, que no supone normalidad, también rechaza, lo que da solidez a la conclusión aunque la compra puntual sea asimétrica.

**Pregunta 1.2.** Redacte la recomendación para gerencia: ¿conviene renovar el contrato marco? Use el intervalo de confianza para dimensionar el ahorro y mencione otro beneficio que se observa en los datos.

> **✅ Respuesta modelo (docente)**
>
> Sí. Con 95 % de confianza, el contrato marco reduce el precio unitario medio entre unos **4 y 7,3 mil pesos por unidad** (alrededor de un 10 % del precio de mercado). Además, con contrato el precio es **mucho más estable** (desviación ≈ 2,8 frente a ≈ 8,3 mil $), lo que facilita el presupuesto y elimina las compras puntuales muy caras (la cola derecha llega a 77 mil $). Para valorar el ahorro total habría que multiplicar la diferencia por el volumen anual de compra y compararlo con los costos del contrato.

---
# Parte 2 (semi-guiada) · ¿Los proveedores entregan acero con la misma resistencia?

Calidad realizó 318 ensayos de tracción sobre bobinas de tres proveedores (Andes, Central y Pacífico). Se quiere saber si la resistencia media es la misma para los tres:

$$H_0:\ \mu_{Andes} = \mu_{Central} = \mu_{Pacífico} \qquad H_1: \text{al menos una media es diferente}$$

El ANOVA de un factor descompone la variabilidad total en la parte **entre grupos** y la parte **dentro de los grupos**: F = CM_entre / CM_dentro.

In [ ]:
proveedores = cargar_csv("IC05_resistencia_proveedores.csv")
display(proveedores.groupby("proveedor")["resistencia_traccion_mpa"].agg(["count", "mean", "std", "min", "max"]).round(2))
sns.boxplot(data=proveedores, x="proveedor", y="resistencia_traccion_mpa")
plt.ylabel("resistencia a la tracción (MPa)"); plt.show()

**2.1** Ajuste el modelo de ANOVA con `smf.ols` y muestre la tabla con `sm.stats.anova_lm`. Guarde el modelo en `modelo_prov`.

In [ ]:
modelo_prov = smf.ols("resistencia_traccion_mpa ~ C(proveedor)", data=proveedores).fit()
tabla_anova = sm.stats.anova_lm(modelo_prov, typ=2)
tabla_anova["CM"] = tabla_anova["sum_sq"] / tabla_anova["df"]
tabla_anova.round(3)

**2.2** Verifique los supuestos del ANOVA: normalidad de los **residuos** (Shapiro y gráfico Q-Q) e igualdad de varianzas entre proveedores (Levene).

In [ ]:
res = modelo_prov.resid
print(f"Shapiro sobre los residuos: p = {stats.shapiro(res).pvalue:.4f}")
grupos = [g["resistencia_traccion_mpa"] for _, g in proveedores.groupby("proveedor")]
print(f"Levene: p = {stats.levene(*grupos).pvalue:.4g}")
print("Desviaciones estándar por grupo:", proveedores.groupby("proveedor")["resistencia_traccion_mpa"].std().round(2).to_dict())
sm.qqplot(res, line="s"); plt.title("Q-Q de los residuos"); plt.show()

**2.3** Como Levene rechaza, complemente con el **ANOVA de Welch** (`anova_oneway(..., use_var="unequal")`) y luego identifique qué pares difieren con Tukey (`pairwise_tukeyhsd`).

In [ ]:
welch_anova = anova_oneway(proveedores["resistencia_traccion_mpa"], proveedores["proveedor"], use_var="unequal")
print(f"ANOVA de Welch: F = {welch_anova.statistic:.2f}   p = {welch_anova.pvalue:.3g}\n")
tukey = pairwise_tukeyhsd(proveedores["resistencia_traccion_mpa"], proveedores["proveedor"], alpha=ALFA)
print(tukey)
tukey.plot_simultaneous(); plt.show()

**Pregunta 2.1.** ¿Qué concluye sobre las medias de los tres proveedores? ¿Qué pares difieren y en cuánto?

> **✅ Respuesta modelo (docente)**
>
> Se rechaza H₀ tanto en el ANOVA clásico como en el de Welch (p ≈ 0): no todos los proveedores tienen la misma resistencia media. Tukey muestra que **Pacífico** está muy por debajo de los otros dos (≈ 47 MPa menos que Andes y ≈ 57 MPa menos que Central). Andes y Central también difieren de forma significativa, pero la diferencia es pequeña (≈ 10 MPa, cerca del 2 %). Los residuos son compatibles con normalidad; la diferencia de varianzas se debe sobre todo a que Pacífico es más disperso, y por eso se reporta también el ANOVA de Welch.

**Pregunta 2.2.** Si la especificación exige una resistencia mínima de 450 MPa, ¿qué recomendaría al área de compras? Apoye su respuesta con la proporción de ensayos por debajo de 450 MPa en cada proveedor (calcúlela).

In [ ]:
(proveedores.assign(bajo_especificacion=proveedores["resistencia_traccion_mpa"] < 450)
            .groupby("proveedor")["bajo_especificacion"].mean().map("{:.1%}".format))

> **✅ Respuesta modelo (docente)**
>
> Pacífico no solo tiene la media más baja sino también la mayor dispersión, por lo que una fracción importante de sus bobinas (más de un tercio) queda por debajo de 450 MPa, mientras que en Andes y Central es muy poco frecuente. Se recomienda reducir la participación de Pacífico o exigirle un plan de mejora con inspección reforzada. Entre Andes y Central la diferencia de 10 MPa es estadísticamente significativa pero poco relevante frente a la especificación: la decisión entre ellos puede basarse en precio y plazo de entrega.

---
# Parte 3 (autónoma) · ¿El tiempo de reparación depende del tipo de equipo?

El área de mantenimiento registró 537 órdenes correctivas de cuatro familias de equipos (Empaque, Transportador, Prensa e Inyección). Quiere saber si el tiempo de reparación es el mismo en todas, para asignar técnicos y repuestos.

Realice un análisis completo: descripción por grupo, revisión de supuestos, la prueba adecuada y las comparaciones por pares. **Pista:** observe la forma de las distribuciones; los tiempos suelen ser asimétricos a la derecha. Considere dos caminos: Kruskal-Wallis sobre los tiempos o ANOVA sobre el **logaritmo** del tiempo.

In [ ]:
reparacion = cargar_csv("IC03_tiempos_reparacion.csv")
orden = reparacion.groupby("tipo_equipo")["tiempo_reparacion_min"].median().sort_values().index
display(reparacion.groupby("tipo_equipo")["tiempo_reparacion_min"].agg(["count", "mean", "median", "std", "skew"]).loc[orden].round(2))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=reparacion, x="tipo_equipo", y="tiempo_reparacion_min", order=orden, ax=ax[0]); ax[0].set_title("Tiempo (min)")
sns.boxplot(data=reparacion.assign(log_t=np.log(reparacion["tiempo_reparacion_min"])), x="tipo_equipo", y="log_t", order=orden, ax=ax[1]); ax[1].set_title("log(tiempo)")
plt.show()

In [ ]:
grupos = {g: d["tiempo_reparacion_min"] for g, d in reparacion.groupby("tipo_equipo")}
print("Shapiro por grupo (tiempo / log tiempo):")
for g, x in grupos.items():
    print(f"  {g:14s} p = {stats.shapiro(x).pvalue:.4f}  /  {stats.shapiro(np.log(x)).pvalue:.4f}")
print(f"Levene tiempo: p = {stats.levene(*grupos.values()).pvalue:.3g}   Levene log: p = {stats.levene(*[np.log(x) for x in grupos.values()]).pvalue:.3f}")
print(f"\nKruskal-Wallis: H = {stats.kruskal(*grupos.values()).statistic:.1f}   p = {stats.kruskal(*grupos.values()).pvalue:.3g}")
m_log = smf.ols("np.log(tiempo_reparacion_min) ~ C(tipo_equipo)", data=reparacion).fit()
display(sm.stats.anova_lm(m_log, typ=2).round(4))
print(pairwise_tukeyhsd(np.log(reparacion["tiempo_reparacion_min"]), reparacion["tipo_equipo"]))

**Pregunta 3.1.** ¿Qué prueba eligió y por qué? ¿Qué concluye? Explique cómo se interpretan las diferencias cuando se trabaja con el logaritmo del tiempo.

> **✅ Respuesta modelo (docente)**
>
> Los tiempos son asimétricos a la derecha en las cuatro familias (Shapiro rechaza) y las varianzas son muy distintas (Levene rechaza), así que el ANOVA sobre los minutos no es adecuado. Hay dos caminos válidos: **Kruskal-Wallis** (no paramétrico) o **ANOVA sobre log(tiempo)**, donde se cumple la normalidad y las varianzas quedan parecidas. Ambos rechazan H₀ con mucha claridad. Todas las parejas difieren; el orden es Inyección > Prensa > Transportador > Empaque. En la escala logarítmica una diferencia de medias d equivale a una **razón** de medianas e^d: por ejemplo, d ≈ 1,06 entre Inyección y Empaque significa que la reparación típica de una inyectora tarda unas 2,9 veces más.

**Pregunta 3.2.** ¿Qué decisión de gestión de mantenimiento sugieren estos resultados?

> **✅ Respuesta modelo (docente)**
>
> Las inyectoras y las prensas concentran las reparaciones más largas y variables: conviene priorizar para ellas el mantenimiento preventivo, tener repuestos críticos en almacén y asignar técnicos especializados. Para planear la capacidad del taller debe usarse un tiempo por familia (no un promedio general) y, dada la asimetría, planear con un percentil alto (por ejemplo el 90 %) en lugar de la media.